# BestBuy Product Classification — GPU Training (Colab)

## Setup
1. **Runtime → Change runtime type → T4 GPU**
2. Run cells in order
3. En la celda 2 vas a tener que subir el CSV manualmente (18MB, va rapido)
4. Al final descarga los 3 result files y ponelos en `src/results/` local

In [ ]:
# 1. Clone repo y setup
!git clone https://github.com/CayllahuaPedro/Sprint-4.git
%cd Sprint-4
import sys, os
sys.path.insert(0, '/content/Sprint-4')
os.makedirs('data/images', exist_ok=True)
os.makedirs('Embeddings', exist_ok=True)
os.makedirs('src/results', exist_ok=True)
print('Setup done. CWD:', os.getcwd())

In [ ]:
# 2. Install dependencies
!pip install -q transformers torch sentence-transformers scikit-learn pandas Pillow requests
# TensorFlow ya viene en Colab

In [ ]:
# 3. Verificar GPU
import tensorflow as tf
import torch
print('TF GPU:', tf.config.list_physical_devices('GPU'))
print('CUDA:', torch.cuda.is_available())

In [ ]:
# 4. Subir el CSV (processed_products_with_images.csv ~18MB)
# Esta celda abre un dialogo para subir el archivo
from google.colab import files
import shutil

print('Selecciona el archivo: data/processed_products_with_images.csv de tu proyecto local')
uploaded = files.upload()
fname = list(uploaded.keys())[0]
shutil.move(fname, 'data/processed_products_with_images.csv')
print('CSV subido correctamente!')

In [ ]:
# 5. Descargar imagenes (500 por clase directo desde URLs)
import pandas as pd
from src.utils import ImageDownloader

df = pd.read_csv('data/processed_products_with_images.csv')
df_sample = df.groupby('class_id', group_keys=False).apply(
    lambda x: x.sample(min(len(x), 500), random_state=42)
)
print(f'Downloading {len(df_sample)} images...')
downloader = ImageDownloader(image_dir='data/images/', image_size=(224, 224))
df_downloaded = downloader.download_images(df_sample, workers=16)
df_downloaded = df_downloaded.dropna(subset=['image_path'])
df_downloaded.to_csv('data/processed_products_with_images.csv', index=False)
print(f'Done. {len(df_downloaded)} images downloaded.')

In [ ]:
# 6. Text embeddings con all-mpnet-base-v2
import torch
from src.nlp_models import HuggingFaceEmbeddings

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using: {device}')

df_products = pd.read_csv('data/processed_products_with_images.csv')
df_products['name_desc'] = (
    df_products['name'].fillna('') + ' ' + df_products['description'].fillna('')
).str.strip()
df_products.to_csv('data/processed_products_with_images.csv', index=False)

model = HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2', device=device)
model.path = 'data/processed_products_with_images.csv'
model.get_embedding_df('name_desc', 'Embeddings/', 'text_embeddings_mpnet.csv')
print('Text embeddings saved!')

In [ ]:
# 7. ViT image embeddings
from src.vision_embeddings_tf import get_embeddings_df

get_embeddings_df(batch_size=32, path='data/images', dataset_name='', backbone='vit_base', directory='Embeddings')
print('ViT embeddings saved!')

In [ ]:
# 8. Fine-tune ViT (con GPU esto tarda ~10 min)
from src.vision_embeddings_tf import fine_tune_model

df_products = pd.read_csv('data/processed_products_with_images.csv')
df_products = df_products.dropna(subset=['image_path'])
df_products['ImageName'] = df_products['image_path'].apply(lambda x: os.path.basename(x))
labels_df = df_products[['ImageName', 'class_id']].rename(columns={'class_id': 'label'})

num_classes = labels_df['label'].nunique()
print(f'Fine-tuning ViT: {num_classes} clases, {len(labels_df)} imagenes...')

ft_embeddings = fine_tune_model(
    backbone='vit_base',
    image_dir='data/images',
    labels_df=labels_df,
    num_classes=num_classes,
    output_dir='Embeddings',
    num_epochs=20,
    batch_size=32,
    lr=1e-4,
    unfreeze_layers=10,
)
print(f'Fine-tuned embeddings shape: {ft_embeddings.shape}')

In [ ]:
# 9. Merge y entrenar clasificadores
import numpy as np
from src.utils import preprocess_data, train_test_split_and_feature_extraction
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

text = pd.read_csv('Embeddings/text_embeddings_mpnet.csv')
images = pd.read_csv('Embeddings/Embeddings_vit_base_finetuned.csv')
df_ft = preprocess_data(text, images, 'image_path', 'ImageName')
df_ft.to_csv('Embeddings/embeddings_mpnet_vit_finetuned.csv', index=False)
print(f'Merged shape: {df_ft.shape}')

train_df, test_df, text_cols, img_cols, label_cols = train_test_split_and_feature_extraction(df_ft)
label_col = label_cols[0]
y_train = train_df[label_col].values
y_test  = test_df[label_col].values

# Text
clf_txt = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', multi_class='multinomial', n_jobs=-1)
clf_txt.fit(train_df[text_cols].values, y_train)
y_pred_txt = clf_txt.predict(test_df[text_cols].values)
txt_acc = accuracy_score(y_test, y_pred_txt)
txt_f1  = f1_score(y_test, y_pred_txt, average='macro')
print(f'Text:  acc={txt_acc:.4f}  f1={txt_f1:.4f}  ({"PASS" if txt_acc>0.85 else "FAIL"})')
pd.DataFrame({'Predictions': y_pred_txt, 'True Labels': y_test}).to_csv('src/results/text_results.csv', index=False)

# Image
clf_img = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', multi_class='multinomial', n_jobs=-1)
clf_img.fit(train_df[img_cols].values, y_train)
y_pred_img = clf_img.predict(test_df[img_cols].values)
img_acc = accuracy_score(y_test, y_pred_img)
img_f1  = f1_score(y_test, y_pred_img, average='macro')
print(f'Image: acc={img_acc:.4f}  f1={img_f1:.4f}  ({"PASS" if img_acc>0.75 else "FAIL"})')
pd.DataFrame({'Predictions': y_pred_img, 'True Labels': y_test}).to_csv('src/results/image_results.csv', index=False)

# Multimodal
X_train_mm = np.hstack([train_df[text_cols].values, train_df[img_cols].values])
X_test_mm  = np.hstack([test_df[text_cols].values,  test_df[img_cols].values])
clf_mm = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', multi_class='multinomial', n_jobs=-1)
clf_mm.fit(X_train_mm, y_train)
y_pred_mm = clf_mm.predict(X_test_mm)
mm_acc = accuracy_score(y_test, y_pred_mm)
mm_f1  = f1_score(y_test, y_pred_mm, average='macro')
print(f'Multi: acc={mm_acc:.4f}  f1={mm_f1:.4f}  ({"PASS" if mm_acc>0.85 else "FAIL"})')
pd.DataFrame({'Predictions': y_pred_mm, 'True Labels': y_test}).to_csv('src/results/multimodal_results.csv', index=False)

print('\n===== FINAL RESULTS =====')
print(f'Text:       acc={txt_acc:.4f}  f1={txt_f1:.4f}  (need >0.85 / >0.80)')
print(f'Image:      acc={img_acc:.4f}  f1={img_f1:.4f}  (need >0.75 / >0.70)')
print(f'Multimodal: acc={mm_acc:.4f}  f1={mm_f1:.4f}  (need >0.85 / >0.80)')

In [ ]:
# 10. Descargar los 3 result files
from google.colab import files

for f in ['src/results/text_results.csv', 'src/results/image_results.csv', 'src/results/multimodal_results.csv']:
    files.download(f)
    print(f'Downloaded: {f}')

print('\nPone estos archivos en Sprint-4/src/results/ local y corre los tests!')